# FLAG 2027 - Notebook 08: Fixing the domain gap (why did VoxCeleb2 not help?)

Notebook 07: +114 MAV-Celeb people helped a lot, +118 VoxCeleb2 people made it slightly worse. Likely cause: **domain shift** (VoxCeleb2 faces/audio are made differently from the challenge data; MAV-Celeb comes from the organizers' own pipeline).

This notebook uses **only features already saved on Drive** (no downloads):
1. **Measure the gap:** can a simple classifier tell which dataset a face / voice comes from?
2. **Fix 1 - per-dataset normalization (`dn`):** scale each dataset with its own mean/std (challenge statistics from TRAINING rows only).
3. **Fix 2 - two-stage training (`2stage`):** first train on external data, then fine-tune on challenge + MAV-Celeb.

Reference results (E2_mav, E3_mav+vox118) are reused from Notebook 07. About 45-75 min. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - load challenge, MAV-Celeb and VoxCeleb2 features
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
VOXF = [dict(np.load(os.path.join(FEATDIR, f))) for f in sorted(os.listdir(FEATDIR)) if f.startswith('vox_') and f.endswith('.npz')]
VOX = {k: np.concatenate([v[k] for v in VOXF]) for k in ['spk', 'clip', 'ecapa', 'wavlm_sv']}
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
print('challenge', len(y), '| MAV', len(MAV['spk']), '| VOX', len(VOX['spk']), 'clips of', len(set(VOX['spk'])), 'people')

In [ ]:
# Cell 3 - MEASURE the domain gap: can a linear classifier tell the datasets apart?
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(0)
def sample(n_rows, k=3000):
    return rng.choice(n_rows, min(k, n_rows), replace=False)
ich, imv, ivx = sample(len(y)), sample(len(MAV['spk'])), sample(len(VOX['spk']))
note('DOMAIN GAP (accuracy telling two datasets apart; 0.50 = indistinguishable, 1.00 = completely different)')
for feat in ['clip', 'ecapa', 'wavlm_sv']:
    ch = FEAT[feat]['train'][ich]
    for other_name, other, io, spk_o in [('MAV-Celeb', MAV, imv, MAV['spk']), ('VoxCeleb2', VOX, ivx, VOX['spk'])]:
        X = np.concatenate([ch, other[feat][io]]); lab = np.r_[np.zeros(len(ch)), np.ones(len(io))]
        groups = np.r_[np.array(['ch_%d' % c for c in y[ich]]), spk_o[io]]   # people-disjoint folds
        acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1)), X, lab, groups=groups, cv=GroupKFold(5)).mean()
        note('  %-9s challenge vs %-10s %.2f' % (feat, other_name, acc))

In [ ]:
# Cell 4 - combined table with a DOMAIN id per row (0 challenge, 1 MAV-Celeb, 2 VoxCeleb2) + model + validation
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(ecapa, wavlm):
    return np.concatenate([l2(ecapa), l2(wavlm)], 1).astype(np.float32)

N_CH, N_MAV, N_VOX = len(y), len(MAV['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_VOX, int)]
F_RAW = np.concatenate([face_of(FEAT['clip']['train']), face_of(MAV['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(FEAT['ecapa']['train'], FEAT['wavlm_sv']['train']), voice_of(MAV['ecapa'], MAV['wavlm_sv']), voice_of(VOX['ecapa'], VOX['wavlm_sv'])])
ROWS = {'mav': N_CH + np.arange(N_MAV), 'mav+vox': N_CH + np.arange(N_MAV + N_VOX), 'vox': N_CH + N_MAV + np.arange(N_VOX)}

def domain_normalize(tr_idx):
    # each dataset scaled by ITS OWN mean/std, computed from its TRAINING rows only.
    # returns normalized copies + the challenge-domain statistics (used for dev/eval data, which is challenge domain)
    F, V = F_RAW.copy(), V_RAW.copy(); ch_stats = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch_stats = st
    return F, V, ch_stats

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()   # HIGHER = same person
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('ready: rows', len(Y_ALL), '| people', len(set(Y_ALL)))

In [ ]:
# Cell 5 - training: per-dataset normalization (dn), optional two-stage, optional DOMAIN-BALANCED batches (bal)
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS, PRE_EP = 16, 8, 2.0, 1e-4, 100, 15, 60, 30

def run_epochs(model, opt, XF, XV, tr, ytr_np, n_ep, rng, vsets=None, patience=PATIENCE, cls_in=None, in_frac=None):
    # cls_in: boolean per class (True = in-domain: challenge or MAV-Celeb); in_frac: share of in-domain people per batch
    ytr = torch.tensor(ytr_np, device=dev)
    mf, sf_, mv, sv = model.norm
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(ytr_np.max() + 1)]
    if in_frac is not None:
        in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]
        n_in = int(round(P_ID * in_frac))
    best, best_ep, best_state, wait = 1e9, n_ep, None, 0
    for ep in range(1, n_ep + 1):
        model.train()
        for _ in range(STEPS):
            if in_frac is None or len(out_cls) == 0:
                chosen = rng.choice(len(pos_by_cls), P_ID, replace=False)
            else:
                chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return best_ep

def train_variant(tr_idx, ext_rows, dn, two_stage, vsets=None, seed=1, max_epochs=MAX_EP, in_frac=None):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, ext_rows])
    if dn:
        F, V, _ = domain_normalize(all_tr)
    else:
        F, V = F_RAW, V_RAW
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    def labels(rows):
        cl = np.unique(Y_ALL[rows]); rm = {c: i for i, c in enumerate(cl)}
        return np.array([rm[c] for c in Y_ALL[rows]]), len(cl)
    fin_rows = np.concatenate([tr_idx, ROWS['mav']]) if two_stage else all_tr
    ytr_np, n_cls = labels(fin_rows)
    cls_in = np.zeros(n_cls, bool)
    cls_in[np.unique(ytr_np[D_ALL[fin_rows] < 2])] = True      # domain 0/1 = challenge / MAV-Celeb
    tr = torch.tensor(fin_rows, device=dev)
    model = FOP(n_cls, XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    if two_stage:
        pre_rows = ext_rows; ypre, n_pre = labels(pre_rows)
        pre = FOP(n_pre, XF.shape[1], XV.shape[1]).to(dev); pre.norm = model.norm
        opt = torch.optim.Adam(pre.parameters(), lr=LR, weight_decay=0.01)
        run_epochs(pre, opt, XF, XV, torch.tensor(pre_rows, device=dev), ypre, PRE_EP, rng)
        st = {k: v for k, v in pre.state_dict().items() if not k.startswith('cls.')}
        model.load_state_dict(st, strict=False)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best_ep = run_epochs(model, opt, XF, XV, tr, ytr_np, max_epochs, rng, vsets, cls_in=cls_in, in_frac=in_frac)
    return model, best_ep, XF, XV

# variant names carry the number of VoxCeleb2 people, so adding a new zip trains a NEW variant
N_VOXP = len(set(VOX['spk']))
VARIANTS = {   # name: (external rows, per-dataset norm, two-stage, in-domain share per batch or None)
    'E2_mav_dn': ('mav', True, False, None),
    'E3_vox%d_dn' % N_VOXP: ('mav+vox', True, False, None),
    'E3_vox%d_dn_bal50' % N_VOXP: ('mav+vox', True, False, 0.50),
    'E3_vox%d_dn_bal75' % N_VOXP: ('mav+vox', True, False, 0.75),
}
print('VoxCeleb2 people available:', N_VOXP, '| variants:', list(VARIANTS))
RESDIR = os.path.join(PROJECT, 'results')
CSV = os.path.join(RESDIR, 'nb08_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
if len(done):   # results of 14 Oct were saved as 'E3_vox_dn' (118 people) -> rename once for clarity
    done['variant'] = done['variant'].replace({'E3_vox_dn': 'E3_vox118_dn', 'E3_vox_dn_2stage': 'E3_vox118_dn_2stage'})
    done.to_csv(CSV, index=False)
t0 = time.time()
for name, (ext, dn, two, frac) in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV = train_variant(tr_idx, ROWS[ext], dn, two, vsets, seed=seed, in_frac=frac)
            e_std, e_gen = [eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-22s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

In [ ]:
# Cell 6 - results table (references E2_mav and E3_mav+vox118 come from Notebook 07)
R7 = pd.read_csv(os.path.join(RESDIR, 'nb07_cv.csv'))
R = pd.concat([R7, pd.read_csv(CSV)], ignore_index=True)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('variant').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                      overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab['gap'] = tab['gender'] - tab['std']
first = [v for v in ['E2_mav', 'E3_mav+vox118', 'E2_mav_dn'] if v in tab.index]
rest = sorted([v for v in tab.index if v not in first], key=lambda v: (int(re.findall(r'\d+', v)[1]) if len(re.findall(r'\d+', v)) > 1 else 0, v))
tab = tab.loc[first + rest].round(2)
note('DOMAIN FIXES / SCALING (K10 features, validation on held-out CHALLENGE people, 5-fold x 3 seeds)')
note(tab.to_string())
BEST = tab['overall'].idxmin()
note('BEST: %s (%.2f)' % (BEST, tab.loc[BEST, 'overall']))

In [ ]:
# Cell 7 - final model for the best variant TRAINED IN THIS RUN (all 70 challenge people, 3 seeds) -> submission ZIP
LOWER_IS_SAME = True
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
cur = tab.loc[[v for v in VARIANTS if v in tab.index]]
BEST_NOW = cur['overall'].idxmin()
note('Best variant of THIS run: %s (%.2f)' % (BEST_NOW, cur.loc[BEST_NOW, 'overall']))
ext, dn, two, frac = VARIANTS[BEST_NOW]
ep = int(np.median(R[R.variant == BEST_NOW]['best_ep']))
all_ch = np.arange(N_CH)
finals = []
for s in SEEDS:
    m, _, XF, XV = train_variant(all_ch, ROWS[ext], dn, two, None, seed=s, max_epochs=ep, in_frac=frac)
    finals.append(m)
_, _, ch_stats = domain_normalize(np.concatenate([all_ch, ROWS[ext]]))   # challenge TRAIN statistics
OUT = '/content/sub_08_' + BEST_NOW
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        split = track + '_' + lang
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
        fx = face_of(FEAT['clip'][split]); vx = voice_of(FEAT['ecapa'][split], FEAT['wavlm_sv'][split])
        if dn:   # dev data is challenge domain -> challenge TRAIN statistics (no test statistics used)
            fx = (fx - ch_stats[0]) / ch_stats[1]; vx = (vx - ch_stats[2]) / ch_stats[3]
        xf = torch.tensor(fx.astype(np.float32), device=dev); xv = torch.tensor(vx.astype(np.float32), device=dev)
        s_ = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        out = -s_ if LOWER_IS_SAME else s_
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, out):
                f.write('%s %.6f\n' % (pid, v))
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, '08_' + BEST_NOW + '.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
note('Submission ZIP saved: %s (%d epochs)' % (ZIP_PATH[len(BASE):], ep))

LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log = log[~log.run.isin(['08_' + v for v in VARIANTS])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '08_' + v,
                     'change': 'ext=%s dn=%s 2stage=%s in-domain share=%s' % VARIANTS[v],
                     'seed': '1,2,3', 'cv_std': tab.loc[v, 'std'], 'cv_gender': tab.loc[v, 'gender'],
                     'cv_overall': tab.loc[v, 'overall'], 'codabench_dev': ''} for v in VARIANTS if v in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)